In [13]:
import pandas as pd
import os

DATA_DIR = '/home/nakamuraroi/kumagai/work/dataset/PatentsViewBulkData'
os.makedirs(DATA_DIR, exist_ok=True)

FILES = {
    "patent": os.path.join(DATA_DIR, "g_patent.tsv"),
    "inventor": os.path.join(DATA_DIR, "g_inventor_disambiguated.tsv"),
    "cpc": os.path.join(DATA_DIR, "g_cpc_current.tsv")
}

In [14]:
import pandas as pd
import numpy as np

# カラム指定による読み込み（メモリ節約）
patent_cols = ['patent_id', 'patent_date']
inventor_cols = ['patent_id', 'inventor_id', 'inventor_sequence']
cpc_cols = ['patent_id', 'cpc_subclass', 'cpc_sequence', 'cpc_type']

# g_patent.tsvの読み込み
df_patent = pd.read_csv(DATA_DIR + '/g_patent.tsv', sep='\t', usecols=patent_cols)
df_patent['patent_date'] = pd.to_datetime(df_patent['patent_date'], errors='coerce')

# 期間フィルタリング（例: 2010年〜2020年）
# 学習の安定性と計算コストのバランスを考慮
start_date = '2010-01-01'
end_date = '2020-12-31'
df_patent = df_patent[(df_patent['patent_date'] >= start_date) & (df_patent['patent_date'] <= end_date)]

# g_cpc_current.tsvの読み込みとフィルタリング
# sequence=0 かつ type=inventional に限定 
df_cpc = pd.read_csv(DATA_DIR + '/g_cpc_current.tsv', sep='\t', usecols=cpc_cols)
df_cpc = df_cpc[(df_cpc['cpc_sequence'] == 0) & (df_cpc['cpc_type'] == 'inventional')]

# g_inventor_disambiguated.tsvの読み込み
df_inventor = pd.read_csv(DATA_DIR + '/g_inventor_disambiguated.tsv', sep='\t', usecols=inventor_cols)

/tmp/ipykernel_1114850/2025574184.py:10: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_patent = pd.read_csv(DATA_DIR + '/g_patent.tsv', sep='\t', usecols=patent_cols)


In [ ]:
df_inventor.head()

,patent_id,inventor_sequence,inventor_id
0,D1006496,0,fl:we_ln:jiang-160
1,12029253,4,fl:ei_ln:baumker-1
2,6584128,0,fl:ri_ln:kroeger-1
3,4789863,0,fl:th_ln:bush-1
4,11161990,1,fl:ma_ln:boudreaux-4


In [ ]:
df_cpc.head()

,patent_id,cpc_sequence,cpc_subclass,cpc_type
0,3950000,0,A63C,inventional
4,3950001,0,A63C,inventional
7,3950002,0,A63C,inventional
11,3950003,0,A63C,inventional
15,3950004,0,B62B,inventional


In [ ]:
df_patent.head()

,patent_id,patent_date
0,10000000,2018-06-19
1,10000001,2018-06-19
2,10000002,2018-06-19
3,10000003,2018-06-19
4,10000004,2018-06-19


In [15]:
# 結合プロセス
# patent_id の型統一
df_inventor['patent_id'] = df_inventor['patent_id'].astype(str)
df_patent['patent_id'] = df_patent['patent_id'].astype(str)
df_cpc['patent_id'] = df_cpc['patent_id'].astype(str)

# Inner Joinにより、発明者情報とCPC情報の両方が揃っている特許のみを残す
df_merged = df_inventor.merge(df_patent, on='patent_id', how='inner')
df_merged = df_merged.merge(df_cpc, on='patent_id', how='inner')

# インデックスのマッピング（文字列ID -> 整数ID）
inv_ids = df_merged['inventor_id'].unique()
cpc_ids = df_merged['cpc_subclass'].unique()

# マッピング辞書の作成
inv_map = {uid: i for i, uid in enumerate(inv_ids)}
cpc_map = {uid: i for i, uid in enumerate(cpc_ids)}

# データフレームへの適用
df_merged['u'] = df_merged['inventor_id'].map(inv_map)
df_merged['v'] = df_merged['cpc_subclass'].map(cpc_map)

# 時間の正規化
# 最初の日付を0とし、年単位（または日単位）の浮動小数点数に変換する
min_date = df_merged['patent_date'].min()
df_merged['t'] = (df_merged['patent_date'] - min_date).dt.days / 365.0

# 最終的なイベントデータの抽出
events = df_merged[['u', 'v', 't']].sort_values('t').reset_index(drop=True)

In [16]:
events

,u,v,t
0,1200165,5,0.000000
1,995754,180,0.000000
2,995755,180,0.000000
3,758603,2,0.000000
4,1068450,52,0.000000
...,...,...,...
9032430,31597,224,10.989041
9032431,944400,224,10.989041
9032432,95565,224,10.989041
9032433,55456,30,10.989041


In [ ]:
%pip install dgl==1.1.3 -f https://data.dgl.ai/wheels/repo.html
!pip install torchdata

  Using cached dgl-2.1.0-cp38-cp38-manylinux1_x86_64.whl (8.6 MB)
  Using cached torchdata-0.8.0-cp38-cp38-manylinux1_x86_64.whl (2.7 MB)


In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ハイパーパラメータ
EMB_DIM = 32
LR = 0.01
EPOCHS = 20
BATCH_SIZE = 4096  # メモリに応じて調整してください

# データ準備 (events DFは前処理済みと仮定)
u_tensor = torch.tensor(events['u'].values, dtype=torch.long).to(device)
v_tensor = torch.tensor(events['v'].values, dtype=torch.long).to(device)
t_tensor = torch.tensor(events['t'].values, dtype=torch.float).to(device)
static_edges = torch.stack([u_tensor, v_tensor])

num_users = len(inv_map)
num_items = len(cpc_map)

# モデル初期化
model = NeuralODEModel(num_users, num_items, EMB_DIM, static_edges).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"Model Initialized. Users: {num_users}, Items: {num_items}")

Using device: cuda


NameError: name 'events' is not defined